# Chapter 6: Generative Models and Dynamics
## Practical: VAE and DDPM on a 2D Ring Dataset

## Part 1: VAE on Ring Dataset

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt

# Generate ring dataset
np.random.seed(42)
N = 5000
radius = 1.0
angle = 2 * np.pi * np.random.rand(N)
x = radius * np.cos(angle) + 0.05 * np.random.randn(N)
y = radius * np.sin(angle) + 0.05 * np.random.randn(N)
data = np.stack([x, y], axis=1)

plt.figure(figsize=(6, 6))
plt.scatter(data[:, 0], data[:, 1], s=1, alpha=0.5)
plt.title("Ring dataset")
plt.axis('equal')
plt.show()

# PyTorch dataset
dataset = TensorDataset(torch.tensor(data, dtype=torch.float32))
loader = DataLoader(dataset, batch_size=128, shuffle=True)

## VAE Model

In [ ]:
latent_dim = 2

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc_mu = nn.Linear(32, latent_dim)
        self.fc_logvar = nn.Linear(32, latent_dim)

    def forward(self, x):
        h = F.relu(self.fc1(x))
        h = F.relu(self.fc2(h))
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(latent_dim, 32)
        self.fc2 = nn.Linear(32, 64)
        self.fc3 = nn.Linear(64, 2)

    def forward(self, z):
        h = F.relu(self.fc1(z))
        h = F.relu(self.fc2(h))
        return self.fc3(h)

class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()

    def reparameterise(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterise(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar

def vae_loss(recon, x, mu, logvar):
    recon_loss = F.mse_loss(recon, x, reduction='sum')
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kl_loss

## Train VAE

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vae = VAE().to(device)
optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)

epochs = 50
loss_history = []
for epoch in range(epochs):
    total_loss = 0
    for batch in loader:
        x = batch[0].to(device)
        recon, mu, logvar = vae(x)
        loss = vae_loss(recon, x, mu, logvar)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(loader.dataset)
    loss_history.append(avg_loss)
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}, Loss: {avg_loss:.4f}")

plt.plot(loss_history)
plt.xlabel('Epoch')
plt.ylabel('Average loss per sample')
plt.title('VAE training loss')
plt.grid(True, alpha=0.3)
plt.show()

## Analyse Latent Space

In [ ]:
vae.eval()
with torch.no_grad():
    x_tensor = torch.tensor(data, dtype=torch.float32).to(device)
    mu, logvar = vae.encoder(x_tensor)
    z = vae.reparameterise(mu, logvar)
    z_np = z.cpu().numpy()
    
    z_prior = torch.randn(1000, latent_dim).to(device)
    gen = vae.decoder(z_prior).cpu().numpy()

angles_true = np.arctan2(data[:, 1], data[:, 0])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].scatter(z_np[:, 0], z_np[:, 1], c=angles_true, cmap='hsv', s=5, alpha=0.6)
axes[0].set_title('Latent space (coloured by true angle)')
axes[0].set_xlabel('z1')
axes[0].set_ylabel('z2')
axes[0].axis('equal')

axes[1].scatter(gen[:, 0], gen[:, 1], s=5, alpha=0.6)
axes[1].set_title('Generated samples (prior)')
axes[1].axis('equal')

axes[2].scatter(data[:, 0], data[:, 1], s=1, alpha=0.3, label='Real')
axes[2].scatter(gen[:, 0], gen[:, 1], s=1, alpha=0.3, label='Generated')
axes[2].set_title('Real vs generated')
axes[2].legend()
axes[2].axis('equal')

plt.tight_layout()
plt.show()

## Part 2: DDPM Implementation on Ring Dataset

In [ ]:
from tqdm import tqdm

# Diffusion hyperparameters
T = 1000
beta_start = 1e-4
beta_end = 0.02

betas = torch.linspace(beta_start, beta_end, T)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

sqrt_alpha_bars = torch.sqrt(alpha_bars)
sqrt_one_minus_alpha_bars = torch.sqrt(1.0 - alpha_bars)

alphas_prev = torch.cat([torch.tensor([1.0]), alphas[:-1]])
posterior_variance = betas * (1.0 - alphas_prev) / (1.0 - alpha_bars)

def get_index_from_list(vals, t, x_shape):
    batch_size = t.shape[0]
    out = vals.gather(-1, t.cpu())
    return out.reshape(batch_size, *((1,) * (len(x_shape) - 1))).to(t.device)

## Noise Predictor Network

In [ ]:
def get_timestep_embedding(timesteps, embedding_dim):
    half_dim = embedding_dim // 2
    emb = torch.log(torch.tensor(10000.0)) / (half_dim - 1)
    emb = torch.exp(torch.arange(half_dim, device=timesteps.device) * -emb)
    emb = timesteps[:, None] * emb[None, :]
    emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
    return emb

class NoisePredictor(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=128, time_emb_dim=64):
        super().__init__()
        self.time_mlp = nn.Sequential(
            nn.Linear(time_emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.fc1 = nn.Linear(input_dim + hidden_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, input_dim)

    def forward(self, x, t):
        t_emb = get_timestep_embedding(t, self.time_mlp[0].in_features)
        t_emb = self.time_mlp(t_emb)
        h = torch.cat([x, t_emb], dim=-1)
        h = F.relu(self.fc1(h))
        h = F.relu(self.fc2(h))
        h = F.relu(self.fc3(h))
        return self.out(h)

## Forward Diffusion

In [ ]:
def forward_diffusion(x0, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)
    sqrt_alpha_bar = get_index_from_list(sqrt_alpha_bars, t, x0.shape)
    sqrt_one_minus_alpha_bar = get_index_from_list(sqrt_one_minus_alpha_bars, t, x0.shape)
    x_t = sqrt_alpha_bar * x0 + sqrt_one_minus_alpha_bar * noise
    return x_t, noise

## Training Loop

In [ ]:
dataset = TensorDataset(torch.tensor(data, dtype=torch.float32))
loader = DataLoader(dataset, batch_size=256, shuffle=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = NoisePredictor().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 100
loss_history = []
for epoch in range(epochs):
    total_loss = 0
    for batch in loader:
        x0 = batch[0].to(device)
        t = torch.randint(0, T, (x0.shape[0],), device=device)
        noise = torch.randn_like(x0)
        xt, noise = forward_diffusion(x0, t, noise)
        predicted_noise = model(xt, t)
        loss = F.mse_loss(predicted_noise, noise)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(loader.dataset)
    loss_history.append(avg_loss)
    if (epoch+1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}, Loss: {avg_loss:.5f}")

plt.plot(loss_history)
plt.xlabel('Epoch')
plt.ylabel('Average loss')
plt.title('DDPM training loss')
plt.grid(True, alpha=0.3)
plt.show()

## Reverse Sampling

In [ ]:
@torch.no_grad()
def sample_ddpm(n_samples=1000, return_trajectory=False):
    model.eval()
    x = torch.randn(n_samples, 2).to(device)
    trajectory = [x.cpu().numpy()] if return_trajectory else None

    for t in reversed(range(1, T)):
        t_tensor = torch.full((n_samples,), t, device=device, dtype=torch.long)
        predicted_noise = model(x, t_tensor)
        
        alpha_t = get_index_from_list(alphas, t_tensor, x.shape)
        alpha_bar_t = get_index_from_list(alpha_bars, t_tensor, x.shape)
        beta_t = get_index_from_list(betas, t_tensor, x.shape)
        alpha_bar_prev = get_index_from_list(alpha_bars, t_tensor-1, x.shape)
        posterior_var = betas[t] * (1.0 - alpha_bar_prev) / (1.0 - alpha_bar_t)

        x = 1.0 / torch.sqrt(alpha_t) * (x - (1 - alpha_t) / torch.sqrt(1 - alpha_bar_t) * predicted_noise)
        if t > 1:
            z = torch.randn_like(x)
            x = x + torch.sqrt(posterior_var) * z
        if return_trajectory and t % 50 == 0:
            trajectory.append(x.cpu().numpy())

    model.train()
    if return_trajectory:
        return x.cpu().numpy(), trajectory
    return x.cpu().numpy()

samples, trajectory = sample_ddpm(n_samples=2000, return_trajectory=True)

## Visualise Results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].scatter(data[:, 0], data[:, 1], s=1, alpha=0.5)
axes[0].set_title('Real data (ring)')
axes[0].axis('equal')

axes[1].scatter(samples[:, 0], samples[:, 1], s=1, alpha=0.5)
axes[1].set_title('DDPM generated samples')
axes[1].axis('equal')

_, traj = sample_ddpm(n_samples=5, return_trajectory=True)
for i in range(5):
    path = [step[i] for step in traj]
    path = np.array(path)
    axes[2].plot(path[:, 0], path[:, 1], 'o-', markersize=2, linewidth=0.5, alpha=0.7)
axes[2].set_title('Reverse trajectory (noise -> data)')
axes[2].axis('equal')

plt.tight_layout()
plt.show()

## Single Sample Trajectory

In [ ]:
single_sample, single_traj = sample_ddpm(n_samples=1, return_trajectory=True)
single_traj = np.array([s[0] for s in single_traj])

plt.figure(figsize=(6, 6))
cmap = plt.cm.viridis
colors = np.linspace(0, 1, len(single_traj))
for i, (x, y) in enumerate(single_traj):
    plt.scatter(x, y, c=[colors[i]], cmap=cmap, s=20, edgecolors='k', linewidth=0.3)
plt.plot(single_traj[:, 0], single_traj[:, 1], 'k--', alpha=0.3)
plt.title('Reverse diffusion path of one sample\n(dark -> noise, light -> data)')
plt.axis('equal')
plt.colorbar(plt.scatter([], [], c=[], cmap=cmap), label='Reverse step')
plt.show()

## Observations

- **VAE**: Learns a smooth latent space capturing the ring structure, but may have "holes".
- **DDPM**: Generates samples that faithfully cover the ring without latent space issues.
- **Reverse trajectories**: Show progression from Gaussian blob to structured ring.
- **Training stability**: DDPM objective is simple regression, very stable.
- **Trade-off**: DDPM requires many sampling steps; VAE is one-pass.